# news_company_hot_issue — 기업을 고르고 시간대를 정하면 기업별 핫이슈 1줄 정리

`news_sector_hot_issue_tmp.ipynb`와 같은 과정을 **고른 기업마다** 한다. 기업마다 정한 시간대의 사건 기사 중 그 기업이 나온 기사에서 **매체 수가 가장 높은 사건 1개**를 골라 OpenAI로 한 줄씩 정리한다.

```
news_extract  ─ llm_source.jsonl에 기사 한 줄(entities: 기사에 나온 기업·인물 등, company_impact: 기업별 영향)
news_track    ─ 같은 줄에 story_id(사건 번호), news_outlets(소식 매체 수) / importance.csv에 story_outlets(사건 매체 수)
   ▼
news_company_hot_issue_tmp.ipynb ─ 시간대 거르기 → 사건 기사만 → 고른 기업마다: 그 기업이 나온 기사 → 사건마다 하나 → 매체 수 1위 → OpenAI 1줄(한 번 호출로 모든 기업)
   ▼  company_hot_issues.jsonl (기업마다 한 줄, 공통 칸을 줄마다 붙임)
SK하이닉스: ~
크래프톤: ~
```

- 이 노트북은 `llm_source.jsonl`, `importance.csv`를 **읽기만** 한다.
- 프롬프트: `prompt/company_hot_issue_prompt.md`(proj/prompt에서 복제).

## 1. 설정

- `COMPANIES`: 요약할 기업의 **종목코드** 목록(`news_collect`·`news_process`의 `COLLECT_COMPANIES`와 같은 목록).
  - 종목코드 → 이름: 검색어 표(`source/terms/test_queries.csv`, 읽기만)에서 `kind`가 `자기 회사`인 검색어를 영어 이름으로 쓴다(예: `000660` → SK하이닉스, "SK hynix").
  - 그 영어 이름을 기사 `entities`의 이름(`name`, `name_en`, `search_terms`, `ticker`)과 맞춰 찾는다. 대소문자·문장부호·끝의 "Inc.", "Corp." 등은 무시한다(예: "SK hynix" = "SK Hynix Inc.").
- `MENTIONS`: 기사에서 그 기업이 어느 정도로 나와야 하는지(`entities`의 `mention`). 값은 `중심`(기사가 주로 다루는 대상) / `실질`(실제 역할이 있음) / `스침`(나열·배경으로만 나옴). 기본은 셋 다(`중심`, `실질`, `스침`).
- `START_KST`, `END_KST`, `TIME_COLUMN`, `MAX_ARTICLES_PER_ISSUE`, `RUN_LLM`: 업종 노트북과 같다.
- 키는 proj 폴더 `.env`의 `OPENAI_API_KEY`이며 값은 출력하지 않는다.
- 순위에 쓰는 매체 수는 **정한 시간대 안의 기사만으로** 다시 센다(2026-11-20 바꿈). `news_track`이 기사마다 남긴 센 출처로 센다. 전체 기간 매체 수는 결과에 `*_all`로 함께 남긴다.
- 처음에 수집 DB(`.env`의 `GNEWS_DB_URL`)에서 시간대 안 기사 수를 세어, `news_extract`가 아직 처리하지 않은 기사가 몇 건인지 알린다(요약은 멈추지 않는다).
- `.env`는 노트북 폴더에서 위로 올라가며 처음 나오는 것을 쓴다.

In [ ]:
import datetime as dt
import json
import re
import urllib.error
import urllib.request
from pathlib import Path

import sqlalchemy as sa

import pandas as pd
from IPython.display import display

# ===== 여기만 바꿔서 쓴다 =====
ROOT = Path.cwd()                      # news_search 폴더(이 노트북이 있는 폴더에서 연다)
PROJ = next((p for p in ROOT.parents if (p / '.env').exists()), ROOT.parents[0])   # .env가 있는 폴더(노트북 폴더에서 위로 올라가며 처음 나오는 곳)
RESULT_DIR = ROOT / 'source' / 'live'  # news_extract·news_track이 함께 쓰는 결과 폴더
COMPANY_PROMPT_PATH = ROOT / 'prompt' / 'company_hot_issue_prompt.md'   # proj/prompt에서 복제
LLM_MODEL = 'gpt-6.1-terra'
RUN_LLM = False          # True여야 OpenAI로 기업별 1줄 정리를 한다
START_KST = '2026-11-18 16:00'   # 시간대 시작(한국 시각, 포함)
END_KST = '2026-11-19 07:30'     # 시간대 끝(한국 시각, 미포함)
TIME_COLUMN = 'published_utc'    # 'published_utc'(발행 시각) 또는 'first_seen_utc'(처음 수집한 시각)
COMPANIES = [            # 요약할 기업(종목코드). news_collect·news_process의 COLLECT_COMPANIES와 같은 목록
    '000660',  # SK하이닉스
    '003490',  # 대한항공
    '004370',  # 농심
    '005490',  # POSCO홀딩스
    '009540',  # HD한국조선해양
    '012450',  # 한화에어로스페이스
    '042700',  # 한미반도체
    '068270',  # 셀트리온
    '087010',  # 펩트론
    '090430',  # 아모레퍼시픽
    '105560',  # KB금융
    '105840',  # 우진
    '114090',  # GKL
    '257720',  # 실리콘투
    '259960',  # 크래프톤
    '352820',  # 하이브
    '383800',  # LX홀딩스
]
MENTIONS = ['중심', '실질', '스침']   # entities의 mention 중 이 값인 기사만(중심 / 실질 / 스침). 지금은 모두 넣는다
MAX_ARTICLES_PER_ISSUE = 3
# ==============================

DB_ENV = 'GNEWS_DB_URL'   # 수집 DB 주소(.env 이름). 시간대 안 기사 중 아직 처리 안 된 수를 알리는 데만 쓴다
ARTICLES_PATH = RESULT_DIR / 'articles.jsonl'       # news_extract가 처리한 기사(DB 확인에 key만 읽는다)
LLM_SOURCE_PATH = RESULT_DIR / 'llm_source.jsonl'   # 읽기만 한다
IMPORTANCE_PATH = RESULT_DIR / 'importance.csv'     # news_track의 매체 수(사건 매체 수 story_outlets를 읽기만 한다)
COMPANY_ISSUES_PATH = RESULT_DIR / 'company_hot_issues.jsonl'   # 정리할 때마다 기업마다 한 줄
TERMS_PATH = ROOT / 'source' / 'terms' / 'test_queries.csv'   # 검색어 표(종목코드 → 이름, 읽기만)
USAGE_PATH = RESULT_DIR / 'llm_usage.csv'           # LLM 호출마다 토큰 수(news_extract·news_track과 함께 씀)
KST = dt.timezone(dt.timedelta(hours=9))
assert ROOT.name == 'news_search', 'news_search 폴더에서 열어야 한다.'

## 2. 작은 도구

업종 노트북과 같은 도구다.
- `env_value`: proj 폴더 `.env`에서 키를 읽는다(값은 출력하지 않는다).
- `ask_openai`: OpenAI Responses API. 오류가 나면 오류 종류·코드만 알린다.

In [ ]:
def env_value(name):
    env_file = PROJ / '.env'
    if env_file.exists():
        for line in env_file.read_text(encoding='utf-8-sig').splitlines():
            key, _, value = line.partition('=')
            if key.strip() == name and value.strip():
                return value.strip().strip('"\'')
    return None


def read_jsonl(path):
    # 한 줄씩 읽는다. 다른 노트북이 쓰는 중인 마지막 줄(덜 쓰인 줄)은 건너뛴다.
    rows = []
    if path.exists():
        for line in path.read_text(encoding='utf-8').splitlines():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return rows


def append_jsonl(path, rows):
    with path.open('a', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')


def append_csv(rows, path):
    if rows:
        data = pd.DataFrame(rows)
        if path.exists():   # 이미 있는 파일의 열 순서에 맞춘다
            data = data.reindex(columns=pd.read_csv(path, nrows=0, encoding='utf-8-sig').columns)
        data.to_csv(path, mode='a', header=not path.exists(), index=False, encoding='utf-8-sig')


class LLMError(Exception):
    pass


def ask_openai(instructions, user_message, text_format):
    request = urllib.request.Request(
        'https://api.openai.com/v1/responses', method='POST',
        headers={'Authorization': f"Bearer {env_value('OPENAI_API_KEY')}", 'Content-Type': 'application/json'},
        data=json.dumps({'model': LLM_MODEL, 'instructions': instructions, 'input': user_message, 'store': False,
                         'reasoning': {'effort': 'medium'}, 'text': {'format': text_format}}).encode('utf-8'))
    try:
        with urllib.request.urlopen(request, timeout=600) as response:
            body = json.loads(response.read().decode('utf-8'))
    except urllib.error.HTTPError as e:
        try:
            err = json.loads(e.read().decode('utf-8', 'replace')).get('error', {})
        except ValueError:
            err = {}
        raise LLMError(f"OpenAI HTTP {e.code} type={err.get('type')} code={err.get('code')}") from None
    text = ''.join(c.get('text', '') for item in body.get('output', []) if item.get('type') == 'message'
                   for c in item.get('content', []))
    return text, body.get('usage') or {}


def log_usage(step, ref, usage):
    append_csv([{'time_utc': dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds'), 'step': step, 'ref': ref,
                 'input_tokens': usage.get('input_tokens'),
                 'cached_tokens': (usage.get('input_tokens_details') or {}).get('cached_tokens'),
                 'output_tokens': usage.get('output_tokens')}], USAGE_PATH)


print('OPENAI_API_KEY:', '설정됨' if env_value('OPENAI_API_KEY') else '미설정')

## 3. 기업마다 사건 고르기

1. `llm_source.jsonl`에서 `TIME_COLUMN`이 시간대 안이고 `story_id`가 있는 기사(사건 기사)만 남긴다. 순위용 매체 수(`news_outlets`, `story_outlets`)는 **시간대 안 기사로만 다시 센다**(2026-11-20 바꿈, 전체 기간 값은 `*_all`).
2. 기업(종목코드 → 영어 이름)마다: `entities` 중 `type`이 `기업`이고 이름이 맞으며 `mention`이 `MENTIONS` 안인 기사만 남긴다. 그 기사의 `company_impact`에서 그 기업의 영향(`impact`)을 가져온다.
3. 사건마다 한 줄: `news_outlets`가 가장 큰 기사(같으면 최근 발행)를 대표로 한다.
4. 사건 순위: `news_outlets` 큰 순 → `story_outlets` 큰 순 → 대표 기사 발행 시각 최근 순. **1위 사건 하나**를 고른다.
5. 시간대 안에 그 기업이 나온 사건 기사가 없으면 그 기업은 건너뛴다(알린다).
6. LLM 입력: 기업마다 대표 기사부터 `MAX_ARTICLES_PER_ISSUE`건(그 기업이 나온 시간대 안 기사)의 제목·매체·발행 시각·short_summary·comparison_summary.

In [ ]:
def to_utc(text):
    t = pd.Timestamp(text)
    return (t.tz_localize(KST) if t.tzinfo is None else t).tz_convert('UTC')


def db_coverage(start_kst, end_kst):
    """DB(수집 결과)에서 시간대 안 기사(거르기 1단계 통과) 중 news_extract가 아직 처리하지 않은 수를 알린다(2026-11-20 더함).
    요약은 멈추지 않는다. 빠진 기사가 있으면 news_extract → news_track을 먼저 돌리면 빠짐없이 요약된다."""
    url = env_value(DB_ENV)
    if not url:
        print(f'DB 확인 건너뜀: .env에 {DB_ENV}가 없다.')
        return None
    col = {'published_utc': 'published_utc', 'first_seen_utc': 'first_seen_utc'}[TIME_COLUMN]
    engine = sa.create_engine(url)
    try:
        with engine.connect() as con:
            keys = {k for (k,) in con.execute(sa.text(f'select key from items where coalesce("drop", \'\') = \'\' and {col} >= :s and {col} < :e and first_seen_utc < :e'),
                                              {'s': to_utc(start_kst).to_pydatetime(), 'e': to_utc(end_kst).to_pydatetime()})}
    except Exception as e:
        print('DB 확인 실패(요약은 계속):', type(e).__name__, str(e)[:150])
        return None
    finally:
        engine.dispose()
    done = {r.get('key') for r in read_jsonl(ARTICLES_PATH)}
    todo = len(keys - done)
    print(f'DB 확인: 시간대 안 수집 기사 {len(keys)}건 중 news_extract 처리 {len(keys & done)}건, 아직 처리 안 됨 {todo}건'
          + (' — news_extract → news_track을 먼저 돌리면 빠짐없이 요약된다' if todo else ''))
    return todo


def window_outlets(rows, in_window):
    """순위용 매체 수를 시간대 안 기사로만 다시 센다(2026-11-20 바꿈).
    news_track이 기사마다 남긴 센 출처(source_key)와 소식 번호(news_id)로, 시간대 안 기사만 세어
    news_outlets(소식 매체 수)·story_outlets(사건 매체 수)를 바꾼다. 전체 기간 값은 news_outlets_all·story_outlets_all로 남긴다."""
    rows = rows.copy()
    rows['news_outlets_all'] = pd.to_numeric(rows['news_outlets'], errors='coerce')
    if IMPORTANCE_PATH.exists():
        imp = pd.read_csv(IMPORTANCE_PATH, dtype=str).drop_duplicates('story_id').set_index('story_id')
        rows['story_outlets_all'] = pd.to_numeric(rows['story_id'].map(imp['story_outlets']), errors='coerce')
    else:
        rows['story_outlets_all'] = float('nan')
    if 'source_key' not in rows or rows['source_key'].isna().all():
        print('주의: llm_source.jsonl에 센 출처(source_key)가 없다 — news_track을 한 번 돌려야 한다. 지금은 전체 기간 매체 수로 순위를 매긴다.')
        rows['story_outlets'] = rows['story_outlets_all']
        return rows
    counted = rows[in_window & rows['source_key'].fillna('').astype(str).ne('')]
    news = counted.groupby(['story_id', 'news_id'])['source_key'].nunique()
    story = counted.groupby('story_id')['source_key'].nunique()
    rows['news_outlets'] = [news.get((s, n)) for s, n in zip(rows['story_id'], rows['news_id'])]
    rows['story_outlets'] = rows['story_id'].map(story)
    return rows


def to_int(values):
    return pd.to_numeric(pd.Series(values), errors='coerce').fillna(0).astype(int)


def int_or0(value):
    try:
        return 0 if value is None or pd.isna(value) else int(value)
    except (TypeError, ValueError):
        return 0


def to_kst_text(utc_text):
    """UTC ISO 시각 → 한국 시각 ISO 문자열(예: 2026-11-16T05:42:00+09:00). 못 읽으면 빈 값."""
    t = pd.to_datetime(utc_text, utc=True, errors='coerce')
    return '' if pd.isna(t) else t.tz_convert(KST).isoformat(timespec='seconds')


def text_or(value, default=''):
    return value if isinstance(value, str) and value else default


SUFFIX = re.compile(r'\b(inc|incorporated|corp|corporation|co|company|ltd|limited|plc|llc|holdings?|group|sa|ag|nv)\b')


def name_key(name):
    """기업 이름 비교용: 소문자, 앞의 the·끝의 Inc./Corp. 등·문장부호·공백을 없앤다."""
    s = (name or '').lower().replace('&', ' and ')
    s = re.sub(r'^the\s+', '', s)
    s = SUFFIX.sub(' ', re.sub(r'[^\w\s]', ' ', s))
    return re.sub(r'\s+', '', s)


def company_names(codes):
    """종목코드 → (한국어 이름, 영어 이름 목록). 검색어 표에서 kind가 '자기 회사'인 검색어(따옴표를 뗀 것)를 영어 이름으로 쓴다."""
    tq = pd.read_csv(TERMS_PATH, dtype=str).fillna('')
    own = tq[tq['kind'] == '자기 회사']
    out, no_name = {}, []
    for code in dict.fromkeys(str(c).strip() for c in codes):   # 같은 코드를 두 번 넣어도 한 번만
        g = own[own['node'] == 'KR:' + code]
        if g.empty:
            no_name.append(code)
            continue
        out[code] = (g['company'].iloc[0], list(dict.fromkeys(q.strip().strip('"') for q in g['query'])))
    if no_name:
        print('주의: 검색어 표에 이름이 없는 종목코드(건너뜀):', no_name)
    return out


def company_entity(row, names):
    """기사(row)의 entities에서 고른 기업(영어 이름 목록 names)과 맞는 기업 entity를 찾는다(MENTIONS 안). 없으면 None."""
    want = {name_key(n) for n in names} - {''}
    for e in row.get('entities') or []:
        if not isinstance(e, dict) or e.get('type') != '기업' or e.get('mention') not in MENTIONS:
            continue
        cand = [e.get('name'), e.get('name_en'), e.get('ticker')] + list(e.get('search_terms') or [])
        if want & {name_key(n) for n in cand if n}:
            return e
    return None


def pick_company_issues(start_kst=None, end_kst=None, companies=None):
    """시간대 안 사건 기사(events)와 기업마다 1위 사건의 대표 기사(top, COMPANIES 순서)를 돌려준다."""
    start_kst, end_kst = start_kst or START_KST, end_kst or END_KST
    companies = companies if companies is not None else COMPANIES
    db_coverage(start_kst, end_kst)
    rows = pd.DataFrame(read_jsonl(LLM_SOURCE_PATH))
    if rows.empty:
        print('llm_source.jsonl이 비어 있다.')
        return rows, rows, {}
    for c in ['entities', 'company_impact', 'story_id', 'news_outlets']:
        if c not in rows:
            rows[c] = None
    if rows['story_id'].isna().all():
        print('주의: story_id가 비어 있다 — news_track을 먼저 돌려야 한다.')
    rows['time'] = pd.to_datetime(rows[TIME_COLUMN], utc=True, format='ISO8601', errors='coerce')
    in_window = rows['time'].ge(to_utc(start_kst)) & rows['time'].lt(to_utc(end_kst))
    # 끝 시각까지 수집된 기사만(2026-11-23): 발행 시각 기준으로 나중에 다시 돌려도, 늦게 수집된 기사가 섞이지 않고 정규 실행(06:20)과 같은 기사를 쓴다
    in_window &= pd.to_datetime(rows['first_seen_utc'], utc=True, format='ISO8601', errors='coerce').lt(to_utc(end_kst))
    rows = window_outlets(rows, in_window)      # 순위용 매체 수: 시간대 안 기사로만
    has_story = rows['story_id'].fillna('').astype(str) != ''
    events = rows[in_window & has_story].copy()
    events['news_outlets'] = pd.to_numeric(events['news_outlets'], errors='coerce').fillna(0).astype(int)
    events['story_outlets'] = to_int(events['story_outlets'])
    events = events.sort_values(['news_outlets', 'story_outlets', 'time'], ascending=[False, False, False]).reset_index(drop=True)

    names = company_names(companies)
    tops, missing, matched = [], [], {}
    for code, (company, en_names) in names.items():
        ents = pd.Series([company_entity(r, en_names) for _, r in events.iterrows()], index=events.index, dtype=object)
        cand = events[ents.notna()]
        matched[company] = cand['key'].tolist()
        if cand.empty:
            missing.append(company)
            continue
        r = cand.iloc[0].copy()                        # 이미 순위 순으로 정렬돼 있어 맨 앞이 1위 사건의 대표 기사
        e = ents[cand.index[0]]
        impact = next((x.get('impact') for x in r['company_impact'] or [] if isinstance(x, dict) and x.get('entity_id') == e.get('id')), None)
        r['company_code'], r['company'], r['entity_name'], r['impact'], r['company_stories'] = code, company, e.get('name_en') or e.get('name'), impact, cand['story_id'].nunique()
        tops.append(r)
    top = pd.DataFrame(tops).reset_index(drop=True) if tops else pd.DataFrame()
    print(f'시간대 {start_kst} ~ {end_kst} (KST, {TIME_COLUMN}) | 시간대 안 기사 {int(in_window.sum())}건, 그중 사건 기사 {len(events)}건 | '
          f'고른 기업 {len(names)}곳 → 정리할 기업 {len(top)}곳'
          + (f' (시간대 안 사건 기사가 없는 기업: {missing})' if missing else ''))
    return events, top, matched


def company_input(events, top, matched):
    """LLM에 넘길 기업 목록(COMPANIES 순서)."""
    items = []
    for _, r in top.iterrows():
        same = events[(events['story_id'] == r['story_id']) & events['key'].isin(matched[r['company']])]
        same = same.head(MAX_ARTICLES_PER_ISSUE)   # 대표 기사가 맨 앞
        items.append({'company': r['company'], 'impact': text_or(r['impact']) or None, 'news_outlets': int(r['news_outlets']),
                      'story_outlets': int(r['story_outlets']),
                      'articles': [{'title': a['title'], 'outlet': text_or(a.get('provider'), a['outlet']),
                                    'published_utc': a['published_utc'], 'short_summary': text_or(a.get('short_summary')),
                                    'comparison_summary': text_or(a.get('comparison_summary'))}
                                   for _, a in same.iterrows()]})
    return items

## 4. 기업별 1줄 정리

- 기업마다 고른 사건을 **한 번의 호출**로 OpenAI에 넘겨 `{"lines": [{"company", "line"}, ...]}`(기업마다 한 줄)을 받는다.
- 받은 줄을 `기업: 문장`으로 보여 주고 `company_hot_issues.jsonl`에 **기업마다 따로 한 줄씩** 남긴다. 줄마다 공통 칸을 붙인다.
  - 공통 칸: 브리핑 날짜 `date`(시간대 끝의 한국 날짜), 생성 시각 `created_kst`(한국), 시간대 `start_kst`·`end_kst`, `time_column`, `model`
  - 기업 칸: `company_code`(종목코드), `company`(한국어 이름), `entity_name`(기사 속 기업 이름), `story_id`(고른 사건), `key`(대표 기사), `published_kst`(대표 기사 발행 시각, 한국), `impact`(대표 기사의 그 기업 영향), `news_outlets`, `story_outlets`, `company_stories`(그 기업이 나온 사건 수), `line`(1줄)
- 받은 기업 이름이 입력과 다르거나 빠지면 알리고, 그 기업은 저장하지 않는다. 정리할 기업이 없으면 OpenAI를 부르지 않는다.

In [ ]:
COMPANY_PROMPT = COMPANY_PROMPT_PATH.read_text(encoding='utf-8')


def company_hot_issues(start_kst=None, end_kst=None, companies=None):
    start_kst, end_kst = start_kst or START_KST, end_kst or END_KST
    events, top, matched = pick_company_issues(start_kst, end_kst, companies)
    if top.empty:
        print('정리할 기업 이슈가 없다.')
        return {}
    display(top[['company_code', 'company', 'entity_name', 'story_id', 'impact', 'news_outlets', 'story_outlets', 'company_stories', 'published_utc', 'title']])
    items = company_input(events, top, matched)
    if not RUN_LLM:
        print('RUN_LLM = False라 OpenAI를 부르지 않는다. LLM에 넘길 입력:')
        print(json.dumps({'companies': items}, ensure_ascii=False, indent=1))
        return {}
    text, usage = ask_openai(COMPANY_PROMPT, 'companies json:\n' + json.dumps({'companies': items}, ensure_ascii=False), {'type': 'json_object'})
    log_usage('company_hot_issue', f'{start_kst}~{end_kst}', usage)
    got = {}
    for x in json.loads(text).get('lines', []):
        if isinstance(x, dict) and x.get('company'):
            got[str(x['company']).strip()] = str(x.get('line', '')).strip().lstrip('-•· ').strip()
    wrong = sorted(set(got) - set(top['company'])) + [c for c in top['company'] if not got.get(c)]
    if wrong:
        print('주의: 입력과 맞지 않거나 빠진 기업(저장하지 않음):', wrong)
    now = dt.datetime.now(dt.timezone.utc)
    common = {'date': to_utc(end_kst).tz_convert(KST).strftime('%Y-%m-%d'),   # 브리핑 날짜(시간대 끝의 한국 날짜)
              'created_kst': now.astimezone(KST).strftime('%Y-%m-%d %H:%M:%S'),   # 생성 시각(한국)
              'start_kst': start_kst, 'end_kst': end_kst, 'time_column': TIME_COLUMN, 'model': LLM_MODEL}
    out = [{**common, 'company_code': r['company_code'], 'company': r['company'], 'entity_name': r['entity_name'], 'story_id': r['story_id'], 'key': r['key'],
            'published_kst': to_kst_text(r['published_utc']), 'impact': text_or(r['impact']) or None, 'news_outlets': int(r['news_outlets']),
            'story_outlets': int(r['story_outlets']), 'company_stories': int(r['company_stories']),
            'news_outlets_all': int_or0(r['news_outlets_all']), 'story_outlets_all': int_or0(r['story_outlets_all']),
            'line': got[r['company']]}
           for _, r in top.iterrows() if got.get(r['company'])]
    append_jsonl(COMPANY_ISSUES_PATH, out)              # 기업마다 따로 한 줄(공통 칸을 줄마다 붙임)
    for r in out:
        print(f"{r['company']}: {r['line']}")
    return {r['company']: r['line'] for r in out}

## 5. 실행

In [ ]:
company_lines = company_hot_issues()